In [ ]:

import os, glob, shutil, subprocess, time, json
T0 = time.time()
KIND, EXTRA, THREADS = 'gm', ['--since', '2026-09-15'], "4"
os.makedirs('/kaggle/working/bin', exist_ok=True)
src = sorted(glob.glob('/kaggle/input/**/corpus-extract', recursive=True))[0]
BIN = '/kaggle/working/bin/corpus-extract'
shutil.copy(src, BIN); os.chmod(BIN, 0o755)
sha = open(os.path.join(os.path.dirname(src), 'SHA256.txt')).read().strip()
print('binary', src, sha, flush=True)
if KIND == 'gm':
    gm = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/episodes.csv', recursive=True))
    assert gm, 'GM dataset not mounted'
    args = ['--gm', gm[0]]
else:
    days = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/manifest.csv', recursive=True)
                   if 'kaggriculture-episodes-20' in p})
    print('daily dirs:', len(days), days[:3], '...', flush=True)
    assert days, 'no daily datasets mounted'
    args = sum((['--daily', d] for d in days), [])
OUT = '/kaggle/working/slim'
cmd = [BIN, '--mode', 'slim', '--out', OUT, '--threads', THREADS] + args + EXTRA
print(' '.join(cmd[:8]), '...', ' '.join(EXTRA), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
files = [f for f in glob.glob(OUT + '/**/*', recursive=True) if os.path.isfile(f)]
tot = sum(os.path.getsize(f) for f in files)
json.dump({'rc': rc, 'secs': round(time.time() - T0, 1), 'bytes': tot, 'files': len(files),
           'kind': KIND, 'extra': EXTRA, 'binary_sha': sha}, open('/kaggle/working/run.json', 'w'), indent=1)
shutil.rmtree('/kaggle/working/bin', ignore_errors=True)
print('RC', rc, 'files', len(files), 'bytes', tot, 'secs', round(time.time() - T0, 1), flush=True)
assert rc == 0, 'corpus-extract failed'
